In [1]:
!pip install ultralytics scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s eta 0:00:00


In [2]:
!tar -xzf train.tar.gz
!tar -xzf test.tar.gz

In [3]:
import h5py
import os
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm


def load_digit_struct(mat_path):
    f = h5py.File(mat_path, 'r')
    ds = f['digitStruct']
    names_ref = ds['name']
    bboxes_ref = ds['bbox']
    records = []

    for i in range(len(names_ref)):
        name = ''.join(chr(c[0]) for c in f[names_ref[i][0]][()])
        bbox = f[bboxes_ref[i][0]]

        def get_attr(attr):
            v = bbox[attr]
            if v.shape[0] == 1:
                return [float(v[0][0])]
            return [float(f[v[j][0]][()].item()) for j in range(v.shape[0])]

        labels  = [int(x) % 10 for x in get_attr('label')]
        tops    = get_attr('top')
        lefts   = get_attr('left')
        heights = get_attr('height')
        widths  = get_attr('width')

        records.append({
            'name': name,
            'labels': labels,
            'tops': tops,
            'lefts': lefts,
            'heights': heights,
            'widths': widths,
        })

    f.close()
    return records


def convert_split(src_dir, out_img_dir, out_lbl_dir):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    records = load_digit_struct(f"{src_dir}/digitStruct.mat")
    skipped = 0

    for rec in tqdm(records, desc=f"Converting {src_dir}"):
        src_path = f"{src_dir}/{rec['name']}"
        if not os.path.exists(src_path):
            skipped += 1
            continue

        img = Image.open(src_path)
        W, H = img.size

        dst_img = f"{out_img_dir}/{rec['name']}"
        os.makedirs(os.path.dirname(dst_img), exist_ok=True)
        shutil.copy(src_path, dst_img)

        lines = []
        for cls, t, l, h, w in zip(
            rec['labels'], rec['tops'], rec['lefts'],
            rec['heights'], rec['widths']
        ):
            cx = max(0.0, min(1.0, (l + w / 2) / W))
            cy = max(0.0, min(1.0, (t + h / 2) / H))
            nw = max(0.001, min(1.0, w / W))
            nh = max(0.001, min(1.0, h / H))
            lines.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        stem = Path(rec['name']).stem
        with open(f"{out_lbl_dir}/{stem}.txt", "w") as f:
            f.write("\n".join(lines))

    print(f"Done. Skipped: {skipped}")


convert_split("train", "svhn_yolo/images/train", "svhn_yolo/labels/train")
convert_split("test",  "svhn_yolo/images/val",   "svhn_yolo/labels/val")

Converting train: 100%|██████████| 33402/33402 [00:13<00:00, 2558.81it/s]


Done. Skipped: 0


Converting test: 100%|██████████| 13068/13068 [00:04<00:00, 3235.81it/s]

Done. Skipped: 0


In [4]:
yaml_content = """
path: svhn_yolo
train: images/train
val: images/val

nc: 10
names: ['0','1','2','3','4','5','6','7','8','9']
"""

with open("svhn.yaml", "w") as f:
    f.write(yaml_content)

In [5]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="svhn.yaml",
    epochs=10,
    imgsz=416,
    batch=128,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    device=0,
    name="svhn_full",
    patience=10,
    workers=4,
    save=True,
    val=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=svhn.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, 

In [9]:
from ultralytics import YOLO
import numpy as np
from tqdm import tqdm
from pathlib import Path
from PIL import Image

model = YOLO("runs/detect/svhn_full/weights/best.pt")

metrics = model.val(data="svhn.yaml", imgsz=416)

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
for i in range(10):
    print(f"  [{i}]  P={metrics.box.p[i]:.3f}  R={metrics.box.r[i]:.3f}  AP50={metrics.box.ap50[i]:.3f}")

def best_match_ious(pred_boxes, gt_boxes, iou_thresh=0.5):
    ious = []
    matched = set()
    for pb in pred_boxes:
        best_iou, best_j = 0, -1
        for j, gb in enumerate(gt_boxes):
            if j in matched:
                continue
            xi1, yi1 = max(pb[0], gb[0]), max(pb[1], gb[1])
            xi2, yi2 = min(pb[2], gb[2]), min(pb[3], gb[3])
            inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
            a1 = (pb[2]-pb[0]) * (pb[3]-pb[1])
            a2 = (gb[2]-gb[0]) * (gb[3]-gb[1])
            iou = inter / (a1 + a2 - inter + 1e-6)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j >= 0:
            ious.append(best_iou)
            if best_iou >= iou_thresh:
                matched.add(best_j)
    return ious

val_img_dir = Path("svhn_yolo/images/val")
val_lbl_dir = Path("svhn_yolo/labels/val")
img_paths = list(val_img_dir.glob("*.png"))[:2000]

all_ious = []

for img_path in tqdm(img_paths, desc="Computing IoU"):
    lbl_path = val_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue

    img = Image.open(img_path)
    W, H = img.size

    gt_boxes = []
    for line in lbl_path.read_text().strip().splitlines():
        parts = list(map(float, line.split()))
        cx, cy, nw, nh = parts[1], parts[2], parts[3], parts[4]
        gt_boxes.append([
            (cx - nw/2) * W, (cy - nh/2) * H,
            (cx + nw/2) * W, (cy + nh/2) * H,
        ])

    results = model.predict(str(img_path), imgsz=416, conf=0.3, verbose=False)
    pred_boxes = [box.xyxy[0].tolist() for box in results[0].boxes]

    if pred_boxes and gt_boxes:
        all_ious.extend(best_match_ious(pred_boxes, gt_boxes))

mean_iou = np.mean(all_ious) if all_ious else 0.0
iou50 = np.mean([v >= 0.50 for v in all_ious]) if all_ious else 0.0
iou75 = np.mean([v >= 0.75 for v in all_ious]) if all_ious else 0.0

print(f"\nMean IoU ({len(img_paths)} images): {mean_iou:.4f}")
print(f"IoU>=0.50: {iou50:.4f}")
print(f"IoU>=0.75: {iou75:.4f}")

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 215.2±120.0 MB/s, size: 6.2 KB)
val: Scanning /content/svhn_yolo/labels/val.cache... 13068 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13068/13068 2.9Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 817/817 10.7it/s 1:16
                   all      13068      26032      0.896      0.854      0.893      0.421
                     0       1687       1744        0.9       0.89      0.913      0.443
                     1       4578       5099       0.87      0.798      0.834       0.34
                     2       3834       4149      0.922       0.89      0.918      0.443
                     3       2696       2882      0.908      0.792      0.874      0.419
                     4       2397       2

Computing IoU: 100%|██████████| 2000/2000 [00:20<00:00, 96.17it/s] 


Mean IoU (2000 images): 0.7182
IoU>=0.50: 0.9402
IoU>=0.75: 0.4789


In [7]:
from ultralytics import YOLO
from pathlib import Path
import cv2

model = YOLO("runs/detect/svhn_full/weights/best.pt")

photos = list(Path("my_photos").glob("*.jpg")) + list(Path("my_photos").glob("*.png"))

for path in photos:
    results = model.predict(str(path), conf=0.3, imgsz=640, iou=0.45)
    for r in results:
        for box in r.boxes:
            cls  = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            print(f"{path.name}  digit={cls}  conf={conf:.2f}  [{x1},{y1},{x2},{y2}]")
        out_path = f"out_{path.name}"
        r.save(filename=out_path)
        print(f"  -> saved {out_path}")


image 1/1 /content/my_photos/Снимок экрана 2026-04-17 142454.png: 384x640 1 1, 1 5, 52.7ms
Speed: 2.2ms preprocess, 52.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Снимок экрана 2026-04-17 142454.png  digit=5  conf=0.59  [737,280,783,357]
Снимок экрана 2026-04-17 142454.png  digit=1  conf=0.48  [703,286,744,356]
  -> saved out_Снимок экрана 2026-04-17 142454.png
